# Extract Questions from All Tests

This notebook extracts questions from all 20 tests using Gemini API and generates a single JSON file.

In [1]:
from google import genai
from google.genai import types
import json
import time
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# API key from environment variable
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Please set GEMINI_API_KEY in .env file")

client = genai.Client(api_key=API_KEY)

# Files
OUTPUT_JSON = Path("../data/extracted_questions.json")

print("✓ Setup complete")

✓ Setup complete


## Load/Save Functions

In [13]:
def load_results():
    """Load existing results"""
    if OUTPUT_JSON.exists():
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_results(data):
    """Save results immediately"""
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

print("✓ Functions ready")

✓ Functions ready


## Test Single Image (Debug)

In [30]:
# Test on one image
test_image = '../data/quiz_sections/test-01/QUESTIONS/test-01_questions.png'

with open(test_image, 'rb') as f:
    image_bytes = f.read()

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=[
        types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
        """Extract all questions from this image in Arabic and French. 
        Return ONLY a JSON array like this:
        [{"question_number": 1, "question_ar": "...", "question_fr": "..."}]
        No markdown, no explanation."""
    ]
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 18.594466857s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type'

## Process All 20 Tests

In [32]:
# Load existing data
all_data = load_results()
print(f"Already extracted: {len(all_data)} tests")

# Process each test
for test_num in range(1, 21):
    test_name = f"test-{test_num:02d}"
    
    # Skip if already done
    if test_name in all_data:
        print(f"⊙ {test_name}: Already extracted")
        continue
    
    # Image path
    img_path = f'../data/quiz_sections/{test_name}/QUESTIONS/{test_name}_questions.png'
    
    try:
        print(f"→ {test_name}: Processing...", end=" ", flush=True)
        
        # Load image
        with open(img_path, 'rb') as f:
            image_bytes = f.read()
        
        # Call Gemini
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
                """Extract all questions from this image in Arabic and French. 
                Return ONLY a JSON array like this:
                [{"question_number": 1, "question_ar": "...", "question_fr": "..."}]
                No markdown, no explanation."""
            ]
        )
        
        # Parse response
        text = response.text.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("\n", 1)[0].replace("```", "").strip()
        
        questions = json.loads(text)
        
        # Save immediately
        all_data[test_name] = questions
        save_results(all_data)
        
        print(f"✓ {len(questions)} questions")
        time.sleep(3)  # Rate limit
        
    except Exception as e:
        print(f"✗ Error: {e}")

print(f"\n{'='*60}")
print(f"DONE: {len(all_data)}/20 tests extracted")
print(f"{'='*60}")

Already extracted: 20 tests
⊙ test-01: Already extracted
⊙ test-02: Already extracted
⊙ test-03: Already extracted
⊙ test-04: Already extracted
⊙ test-05: Already extracted
⊙ test-06: Already extracted
⊙ test-07: Already extracted
⊙ test-08: Already extracted
⊙ test-09: Already extracted
⊙ test-10: Already extracted
⊙ test-11: Already extracted
⊙ test-12: Already extracted
⊙ test-13: Already extracted
⊙ test-14: Already extracted
⊙ test-15: Already extracted
⊙ test-16: Already extracted
⊙ test-17: Already extracted
⊙ test-18: Already extracted
⊙ test-19: Already extracted
⊙ test-20: Already extracted

DONE: 20/20 tests extracted


## View Results

In [6]:
# Load and display
data = load_results()

print(f"Total tests: {len(data)}")
print(f"Total questions: {sum(len(q) for q in data.values())}")

print("\nTests:")
for test, questions in sorted(data.items()):
    print(f"  {test}: {len(questions)} questions")

# Show sample
if data:
    first = list(data.keys())[0]
    print(f"\nSample from {first}:")
    print(json.dumps(data[first][:1], ensure_ascii=False, indent=2))

Total tests: 18
Total questions: 108

Tests:
  test-01: 6 questions
  test-02: 6 questions
  test-03: 6 questions
  test-04: 6 questions
  test-05: 6 questions
  test-06: 6 questions
  test-07: 6 questions
  test-08: 6 questions
  test-09: 6 questions
  test-10: 6 questions
  test-11: 6 questions
  test-12: 6 questions
  test-13: 6 questions
  test-14: 6 questions
  test-15: 6 questions
  test-16: 6 questions
  test-17: 6 questions
  test-18: 6 questions

Sample from test-01:
[
  {
    "question_number": 1,
    "question_ar": "بأي عامل ترتبط مسافة الأمان؟",
    "question_fr": "De quels facteurs dépend la distance de sécurité ?"
  }
]
